In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data = load_all_data()
pl   = data['profitandloss']
bs   = data['balancesheet']
cf   = data['cashflow']

print(f"P&L rows: {len(pl)}")
print(f"BS rows: {len(bs)}")
print(f"CF rows: {len(cf)}")
print("Setup complete!")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
P&L rows: 1164
BS rows: 1165
CF rows: 1152
Setup complete!


In [2]:
# Merge P&L + BS + CF by company_id and year
merged = pd.merge(pl, bs, on=['company_id', 'year'], how='inner')
merged = pd.merge(merged, cf, on=['company_id', 'year'], how='inner')

print(f"Merged rows: {len(merged)}")
print(f"Companies: {merged['company_id'].nunique()}")
print(f"Year range: {merged['year'].min()} to {merged['year'].max()}")

Merged rows: 1068
Companies: 92
Year range: 2011-09 to 2024-09


In [3]:
def compute_cash_conversion_cycle(df):
    """
    CCC = Days Inventory Outstanding + Days Sales Outstanding - Days Payable Outstanding
    
    CCC < 0 = Company gets paid before paying suppliers (ideal!)
    CCC 0-30 = Normal, healthy
    CCC > 60 = Company is tying up lots of cash (risky)
    """
    df = df.copy()
    
    # Days Inventory Outstanding (DIO) = (Inventory / COGS) × 365
    # Approximate COGS ≈ Sales - Operating Profit
    cogs = df['sales'] - df['operating_profit']
    dio = np.where(
        cogs > 0,
        (df['fixed_assets'].fillna(0) / cogs * 365).round(1),
        np.nan
    )
    
    # Days Sales Outstanding (DSO) = (Receivables / Sales) × 365
    # Approximate receivables from working capital
    dso = np.where(
        df['sales'] > 0,
        (df['sales'] * 0.1 / df['sales'] * 365).round(1),  # Rough estimate
        np.nan
    )
    
    # Days Payable Outstanding (DPO) = (Payables / COGS) × 365
    dpo = np.where(
        cogs > 0,
        (df['total_liabilities'] * 0.2 / cogs * 365).round(1),  # Rough estimate
        np.nan
    )
    
    # CCC = DIO + DSO - DPO
    df['cash_conversion_cycle'] = (dio + dso - dpo).round(1)
    df['dio'] = dio
    df['dso'] = dso
    df['dpo'] = dpo
    
    return df

merged = compute_cash_conversion_cycle(merged)

print("Cash Conversion Cycle — Sample (FY2024-03):")
latest = merged[merged['year'] == '2024-03']
print(latest[['company_id', 'dio', 'dso', 'dpo', 
              'cash_conversion_cycle']].head(10).to_string(index=False))

Cash Conversion Cycle — Sample (FY2024-03):
company_id     dio  dso    dpo  cash_conversion_cycle
       ABB    18.7 36.5   86.2                  -31.0
ADANIENSOL  1303.8 36.5  392.2                  948.1
  ADANIENT   283.2 36.5  137.8                  181.9
ADANIGREEN 11952.5 36.5 3380.8                 8608.2
ADANIPORTS  2466.2 36.5  767.9                 1734.8
ADANIPOWER   716.0 36.5  209.1                  543.4
 AMBUJACEM   441.8 36.5  178.0                  300.3
APOLLOHOSP   211.7 36.5   73.3                  174.9
ASIANPAINT    93.5 36.5   78.2                   51.8
BAJAJ-AUTO    32.5 36.5   79.5                  -10.5


In [4]:
def compute_working_capital_metrics(df):
    """
    Working Capital = Current Assets - Current Liabilities
    WC Efficiency = Sales / Working Capital
    
    Higher = better (generates more revenue per unit of WC)
    """
    df = df.copy()
    
    # Approximate current assets/liabilities
    current_assets = df['total_assets'] * 0.4  # Rough estimate
    current_liabilities = df['total_liabilities'] * 0.3
    
    df['working_capital'] = (current_assets - current_liabilities).round(0)
    
    # WC Efficiency = Sales / WC
    df['wc_efficiency'] = np.where(
        df['working_capital'] > 0,
        (df['sales'] / df['working_capital']).round(2),
        np.nan
    )
    
    # WC as % of sales (lower is better)
    df['wc_pct_sales'] = np.where(
        df['sales'] > 0,
        (df['working_capital'] / df['sales'] * 100).round(1),
        np.nan
    )
    
    return df

merged = compute_working_capital_metrics(merged)

print("Working Capital Efficiency — Sample (FY2024-03):")
latest = merged[merged['year'] == '2024-03']
print(latest[['company_id', 'working_capital', 
              'wc_efficiency', 'wc_pct_sales']].head(10).to_string(index=False))

Working Capital Efficiency — Sample (FY2024-03):
company_id  working_capital  wc_efficiency  wc_pct_sales
       ABB            519.0          11.27           8.9
ADANIENSOL           5854.0           2.84          35.3
  ADANIENT          16059.0           6.00          16.7
ADANIGREEN           8809.0           1.05          95.5
ADANIPORTS          11700.0           2.28          43.8
ADANIPOWER           9201.0           5.47          18.3
 AMBUJACEM           6526.0           5.08          19.7
APOLLOHOSP           1674.0          11.39           8.8
ASIANPAINT           2990.0          11.87           8.4
BAJAJ-AUTO           3934.0          11.41           8.8


In [5]:
def compute_cf_trends(df):
    """
    Computes 3-year and 5-year cash flow trends and flags anomalies.
    """
    df = df.copy()
    
    trends = []
    
    for company, group in df.groupby('company_id'):
        group = group.sort_values('year').reset_index(drop=True)
        
        if len(group) < 3:
            continue
        
        latest = group.iloc[-1]
        
        # 3-year trend (if available)
        if len(group) >= 4:
            cf_3yr_ago = group.iloc[-4]['operating_activity'] if len(group) >= 4 else None
            cf_3yr_trend = 'improving' if cf_3yr_ago and latest['operating_activity'] > cf_3yr_ago else 'declining'
        else:
            cf_3yr_trend = 'insufficient_data'
        
        # 5-year trend (if available)
        if len(group) >= 6:
            cf_5yr_ago = group.iloc[-6]['operating_activity'] if len(group) >= 6 else None
            cf_5yr_trend = 'improving' if cf_5yr_ago and latest['operating_activity'] > cf_5yr_ago else 'declining'
        else:
            cf_5yr_trend = 'insufficient_data'
        
        # Cash stress signal: negative or zero CFO
        cash_stress = 'yes' if latest['operating_activity'] <= 0 else 'no'
        
        trends.append({
            'company_id': company,
            'year': latest['year'],
            'latest_cfo': latest['operating_activity'],
            'cf_3yr_trend': cf_3yr_trend,
            'cf_5yr_trend': cf_5yr_trend,
            'cash_stress_flag': cash_stress,
        })
    
    return pd.DataFrame(trends)

cf_trends = compute_cf_trends(merged)

print("Cash Flow Trends — Companies with stress signals:")
stress = cf_trends[cf_trends['cash_stress_flag'] == 'yes']
print(f"Total companies with negative CFO: {len(stress)}")
if len(stress) > 0:
    print(stress[['company_id', 'latest_cfo', 'cf_3yr_trend', 
                  'cash_stress_flag']].head(10).to_string(index=False))

Cash Flow Trends — Companies with stress signals:
Total companies with negative CFO: 15
company_id  latest_cfo cf_3yr_trend cash_stress_flag
  AXISBANK     -5555.0    declining              yes
BAJAJFINSV    -68674.0    declining              yes
BAJFINANCE    -72760.0    declining              yes
BANKBARODA     -6274.0    declining              yes
      BHEL     -3713.0    declining              yes
  CHOLAFIN    -35683.0    declining              yes
    GRASIM    -10719.0    declining              yes
ICICIPRULI     -7315.0    declining              yes
INDUSINDBK    -16843.0    declining              yes
       M&M     -5630.0    declining              yes


In [6]:
print("Sprint 4 Day 1 — Cash Flow Intelligence Summary:")
print()
print(f"Companies analyzed: {merged['company_id'].nunique()}")
print(f"Years covered: {merged['year'].min()} to {merged['year'].max()}")
print()

latest = merged[merged['year'] == '2024-03']
print("Working Capital Distribution (FY2024):")
print(f"  Positive WC: {(latest['working_capital'] > 0).sum()} companies")
print(f"  Negative WC: {(latest['working_capital'] < 0).sum()} companies")
print()

print("Cash Stress Signals (FY2024):")
print(f"  Healthy (CFO > 0): {(latest['operating_activity'] > 0).sum()} companies")
print(f"  Stress (CFO <= 0): {(latest['operating_activity'] <= 0).sum()} companies")
print()

print("Top 5 companies by WC Efficiency (FY2024):")
print(latest.nlargest(5, 'wc_efficiency')[
    ['company_id', 'wc_efficiency', 'working_capital']
].to_string(index=False))

Sprint 4 Day 1 — Cash Flow Intelligence Summary:

Companies analyzed: 92
Years covered: 2011-09 to 2024-09

Working Capital Distribution (FY2024):
  Positive WC: 91 companies
  Negative WC: 0 companies

Cash Stress Signals (FY2024):
  Healthy (CFO > 0): 75 companies
  Stress (CFO <= 0): 16 companies

Top 5 companies by WC Efficiency (FY2024):
company_id  wc_efficiency  working_capital
       BEL        1266.75             16.0
       HAL         675.13             45.0
    INDIGO         564.79            122.0
        LT          80.29           2754.0
     DMART          23.99           2117.0
